In [ ]:
import numpy as np
import pandas as pd
import glob
import os
import io
import re

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Dictionary to store datasets
dataframes = {}

# Read each uploaded file
for filename in uploaded.keys():
    df = pd.read_csv(io.BytesIO(uploaded[filename]))
    dataframes[filename] = df
    print(f"\n--- {filename} ---")
    print(df.head())  # Print first 5 rows of each file

In [ ]:
filename = list(uploaded.keys())[0]

# Read it (CSV assumed)
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(df.info())
print(df.head())
df

In [ ]:
# Store cleaned DataFrames
cleaned_data = []
all_data = {}

In [ ]:
# Extract base name and inside [...]
base = df["Variable Name"].str.extract(r'^([^\[]+)', expand=False)
inside = df["Variable Name"].str.extract(r'\[(.*)\]', expand=False)

# Split by commas
parts = inside.str.split(",", n=3, expand=True)
parts.columns = ["year", "stage", "resource", "region"]

# Add back to dataframe
out = pd.concat([df, base.rename("base_var"), parts], axis=1)
print(out.head())

In [ ]:
# Ensure we have a 'Variable' column to parse
if "Variable" not in df.columns and "Variable Name" in df.columns:
    df = df.rename(columns={"Variable Name": "Variable"})

In [ ]:
def parse_variable_components(df, var_col="Variable"):
    out = df.copy()
    out[var_col] = out[var_col].astype(str)

    # Base type and bracket content
    out["Variable_Type"]  = out[var_col].str.extract(r"^([^\[]+)", expand=False).str.strip()
    out["Bracket_Content"] = out[var_col].str.extract(r"\[(.*?)\]", expand=False)

    # Split into list
    out["Component_List"] = out["Bracket_Content"].apply(
        lambda s: [c.strip() for c in s.split(",")] if isinstance(s, str) else np.nan
    )

    # Pre-create target columns
    for c in ["Year","Stage","Tech_In","Tech_Out","Region","Region_From","Region_To"]:
        if c not in out.columns:
            out[c] = pd.NA

    def parse_row(row):
        comps = row["Component_List"]
        if not isinstance(comps, list):
            return row

        n = len(comps)
        vtype = (row["Variable_Type"] or "").lower()

        if n == 1:
            # e.g., capacity[US]
            row["Region"] = comps[0]

        elif n == 4:
            # e.g., prod_mine[2020,mine,brine,US]
            row["Year"]  = pd.to_numeric(comps[0], errors="coerce")
            row["Stage"] = comps[1]
            if "mine" in vtype:
                row["Tech_In"]  = "Earth"
                row["Tech_Out"] = comps[2]
            else:
                row["Tech_In"]  = comps[2]
                row["Tech_Out"] = comps[2]
            row["Region"] = comps[3]

        elif n == 5:
            row["Year"]  = pd.to_numeric(comps[0], errors="coerce")
            row["Stage"] = comps[1]

            if vtype.startswith(("import",)):   # imports
                # Year, Stage, Tech_In, Tech_Out, Region_To
                row["Tech_In"]   = comps[2]
                row["Tech_Out"]  = comps[3]
                row["Region_To"] = comps[4]

            elif vtype.startswith(("export","trade","ship","flow")):  # exports
                # Year, Stage, Tech_In, Tech_Out, Region_From
                row["Tech_In"]     = comps[2]
                row["Tech_Out"]    = comps[3]
                row["Region_From"] = comps[4]

            else:  # non-trade 5-tuple
                row["Tech_In"]  = comps[2]
                row["Tech_Out"] = comps[3]
                row["Region"]   = comps[4]

        elif n == 6:
            row["Year"]  = pd.to_numeric(comps[0], errors="coerce")
            row["Stage"] = comps[1]

            if vtype.startswith(("import",)):   # imports
                # Year, Stage, Tech_In, Tech_Out, Region_To, Region_From
                row["Tech_In"]     = comps[2]
                row["Tech_Out"]    = comps[3]
                row["Region_To"]   = comps[4]
                row["Region_From"] = comps[5]

            elif vtype.startswith(("export","trade","ship","flow")):  # exports
                # Year, Stage, Tech_In, Tech_Out, Region_From, Region_To
                row["Tech_In"]     = comps[2]
                row["Tech_Out"]    = comps[3]
                row["Region_From"] = comps[4]
                row["Region_To"]   = comps[5]

            else:  # non-trade 6-tuple
                row["Tech_In"]     = comps[2]
                row["Tech_Out"]    = comps[3]
                row["Region_From"] = comps[4]
                row["Region_To"]   = comps[5]

        # ignore other lengths silently
        return row

    out = out.apply(parse_row, axis=1)

    # Clean up helpers
    return out.drop(columns=["Bracket_Content","Component_List"])


In [ ]:
#apply the parser and preview
parsed = parse_variable_components(df)
print(parsed.head(10))

In [ ]:
parsed = parse_variable_components(df)

# Replace 'Variable' with base type only, but keep the full name (no suffix removal)
cleaned = parsed.copy()
cleaned["Variable"] = cleaned["Variable_Type"]
cleaned = cleaned.drop(columns=["Variable_Type"])

# Show first 20 rows
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
cleaned.head(20)

In [ ]:
# Save cleaned dataframe
output_path = "cleaned_variables.csv"
cleaned.to_csv(output_path, index=False)

print(f"✅ Cleaned dataset saved as: {output_path}")

In [ ]:
# Filter for import or export variables
trade_df = cleaned[cleaned["Variable"].str.lower().isin(["import_pack", "export_pack"])]

# Display first 100 rows
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)
trade_df.head(100)

In [ ]:
# ======= FINAL ANALYSIS ON cleaned_variables.csv (prints + saves) =======
import pandas as pd
import numpy as np

# --- Load cleaned dataset ---
df = pd.read_csv("cleaned_variables.csv")

# --- Standardize columns & types ---
# Ensure required columns exist (some may not be used but keep for completeness)
for col in ["Variable", "Year", "Stage", "Region", "Solution Value"]:
    if col not in df.columns:
        df[col] = pd.NA

# Clean strings
for c in ["Variable", "Stage", "Region"]:
    df[c] = df[c].astype(str).str.strip()

# Coerce numeric safely (handles "1,234.5", spaces, Inf)
def to_num(s):
    return (
        s.astype(str)
         .str.strip()
         .str.replace(",", "", regex=False)
         .replace({"inf": np.inf, "Inf": np.inf, "INF": np.inf, "nan": np.nan, "None": np.nan})
         .pipe(pd.to_numeric, errors="coerce")
    )

df["Year"] = to_num(df["Year"])
df["Solution Value"] = to_num(df["Solution Value"])

# Container for outputs to save
analysis_results = {}

# === PRODUCTION ===
prod = df[df["Variable"].str.startswith("prod_") & df["Year"].notna()].copy()

if not prod.empty:
    production_summary = (
        prod.groupby(["Stage","Region","Year"], as_index=False)["Solution Value"]
           .sum()
           .rename(columns={"Solution Value":"Total_Production"})
           .sort_values(["Stage","Region","Year"])
    )
    production_pivot = (
        production_summary
          .pivot_table(index=["Stage","Region"], columns="Year",
                       values="Total_Production", fill_value=0)
          .sort_index()
    )
    prod_stage_totals = (
        prod.groupby("Stage")["Solution Value"]
            .sum()
            .sort_values(ascending=False)
            .rename("Total_Production")
    )
    prod_reg_stage = (
        prod.groupby(["Region","Stage"])["Solution Value"]
            .sum()
            .unstack("Stage").fillna(0)
            .sort_index()
    )

    print("\n📈 PRODUCTION BY STAGE & REGION (wide):")
    print(production_pivot)

    print("\n📊 TOTAL PRODUCTION BY STAGE:")
    print(prod_stage_totals)

    print("\n🌍 PRODUCTION BY STAGE FOR EACH REGION (totals):")
    print(prod_reg_stage)

    analysis_results.update({
        "prod_summary_long": production_summary,
        "prod_pivot_wide": production_pivot,
        "prod_totals_by_stage": prod_stage_totals,
        "prod_region_x_stage_totals": prod_reg_stage
    })
else:
    print("\n⚠️ No rows matched Variable starting with 'prod_' and non-null Year.")

# === ACTIVE FACILITIES ===
active = df[df["Variable"].str.startswith("active_") & df["Year"].notna()].copy()

if not active.empty:
    active_summary = (
        active.groupby(["Stage","Region","Year"], as_index=False)["Solution Value"]
              .sum()
              .rename(columns={"Solution Value":"Total_Active"})
              .sort_values(["Stage","Region","Year"])
    )
    active_pivot = (
        active_summary
          .pivot_table(index=["Stage","Region"], columns="Year",
                       values="Total_Active", fill_value=0)
          .sort_index()
    )
    active_stage_totals = (
        active.groupby("Stage")["Solution Value"]
              .sum()
              .sort_values(ascending=False)
              .rename("Total_Active")
    )
    active_reg_stage = (
        active.groupby(["Region","Stage"])["Solution Value"]
              .sum()
              .unstack("Stage").fillna(0)
              .sort_index()
    )

    print("\n🏭 ACTIVE FACILITIES BY STAGE & REGION (wide):")
    print(active_pivot)

    print("\n🔢 TOTAL ACTIVE FACILITIES BY STAGE:")
    print(active_stage_totals)

    print("\n🏢 ACTIVE FACILITIES BY STAGE FOR EACH REGION (totals):")
    print(active_reg_stage)

    analysis_results.update({
        "active_summary_long": active_summary,
        "active_pivot_wide": active_pivot,
        "active_totals_by_stage": active_stage_totals,
        "active_region_x_stage_totals": active_reg_stage
    })
else:
    print("\n⚠️ No rows matched Variable starting with 'active_' and non-null Year.")

# === SAVE EVERYTHING TO EXCEL (uses openpyxl; no extra install needed) ===
out_xlsx = "analysis_cleaned.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    for key, obj in analysis_results.items():
        if isinstance(obj, pd.Series):
            obj.to_frame().to_excel(writer, sheet_name=key)
        else:
            obj.to_excel(writer, sheet_name=key)

print(f"\n✅ Analysis printed above and saved to {out_xlsx}")


In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
library(readr)
library(dplyr)
library(ggplot2)
library(stringr)
library(tidyr)

# Read cleaned data
df <- read_csv("cleaned_variables.csv", show_col_types = FALSE)

# Rename "Solution Value" -> "Value" if it exists
if ("Solution Value" %in% names(df) && !("Value" %in% names(df))) {
  df <- df %>% rename(Value = `Solution Value`)
}

# Coerce Year to integer
df <- df %>%
  mutate(Year = suppressWarnings(as.integer(Year)))


In [ ]:
%%R
prod_sum <- df %>%
  filter(str_starts(Variable, "prod_"), !is.na(Year), !is.na(Value)) %>%
  group_by(Region, Stage, Year) %>%
  summarize(Total_Production = sum(Value, na.rm = TRUE), .groups = "drop")

# Line plot: all stages on same axes per region
p_prod <- ggplot(prod_sum, aes(x = Year, y = Total_Production, color = Stage, group = Stage)) +
  geom_line() +
  geom_point(size = 1.5) +
  facet_wrap(~ Region, scales = "free_y") +
  labs(title = "Total Production by Stage over Time (faceted by Region)",
       x = "Year", y = "Total Production") +
  theme_minimal(base_size = 12) +
  theme(legend.position = "bottom")

print(p_prod)

# (optional) save
ggsave("prod_by_stage_over_time.png", p_prod, width = 14, height = 8, dpi = 150)


In [ ]:
%%R
active_sum <- df %>%
  filter(str_starts(Variable, "active_"), !is.na(Year), !is.na(Value)) %>%
  group_by(Region, Stage, Year) %>%
  summarize(Total_Active = sum(Value, na.rm = TRUE), .groups = "drop")

p_active <- ggplot(active_sum, aes(x = Year, y = Total_Active, color = Stage, group = Stage)) +
  geom_line() +
  geom_point(size = 1.5) +
  facet_wrap(~ Region, scales = "free_y") +
  labs(title = "Active Facilities by Stage over Time (faceted by Region)",
       x = "Year", y = "Total Active Facilities") +
  theme_minimal(base_size = 12) +
  theme(legend.position = "bottom")

print(p_active)

# (optional) save
ggsave("active_by_stage_over_time.png", p_active, width = 14, height = 8, dpi = 500)


In [ ]:
%%R
ggsave("prod_by_stage_over_time.png", p_prod, width = 14, height = 8, dpi = 300)
ggsave("active_by_stage_over_time.png", p_active, width = 14, height = 8, dpi = 300)


In [ ]:
%%R
library(dplyr)
library(stringr)

# Ensure Value/Year exist (safe if you already did this)
df <- readr::read_csv("cleaned_variables.csv", show_col_types = FALSE)
if ("Solution Value" %in% names(df) && !("Value" %in% names(df))) {
  df <- df %>% rename(Value = `Solution Value`)
}
df <- df %>% mutate(Year = suppressWarnings(as.integer(Year)))

# --- IMPORT totals: destination = Region_To; sum across Region_From for each Stage + Year
import_sum <- df %>%
  filter(str_starts(Variable, "import_"),
         !is.na(Year), !is.na(Value), !is.na(Region_To)) %>%
  group_by(Region = Region_To, Stage, Year) %>%
  summarize(Total_Import = sum(Value, na.rm = TRUE), .groups = "drop")

# Peek
print(head(import_sum, 10))


In [ ]:
%%R
library(ggplot2)

# Make plots larger in Colab
options(repr.plot.width = 14, repr.plot.height = 8)

p_import <- ggplot(import_sum, aes(x = Year, y = Total_Import, color = Stage, group = Stage)) +
  geom_line() +
  geom_point(size = 1.5) +
  facet_wrap(~ Region, scales = "free_y") +
  labs(title = "Total Imports by Stage over Time (Destination = Region_To)",
       subtitle = "Summed across all Region_From origins",
       x = "Year", y = "Total Imports") +
  theme_minimal(base_size = 12) +
  theme(legend.position = "bottom")

print(p_import)

# Optional: save high-res images
ggsave("imports_by_stage_over_time.png", p_import, width = 14, height = 8, dpi = 300)


In [ ]:
# %%R
# library(readr)
# library(dplyr)
# library(stringr)
# library(ggplot2)
# library(scales)

# # Read cleaned data
# df <- read_csv("cleaned_variables.csv", show_col_types = FALSE)

# # Normalize metric column
# if ("Solution Value" %in% names(df) && !("Value" %in% names(df))) {
#   df <- df %>% rename(Value = `Solution Value`)
# }

# # Basic cleaning
# df <- df %>%
#   mutate(
#     Year  = suppressWarnings(as.integer(Year)),
#     Stage = as.character(Stage),
#     Region = as.character(Region)
#   )

# # Make plots large in Colab
# options(repr.plot.width = 14, repr.plot.height = 8)


## Create table for Model Validity Check

In [ ]:
# unify value col
if "Value" not in df.columns and "Solution Value" in df.columns:
    df = df.rename(columns={"Solution Value": "Value"})

# keep only needed cols
keep = ["Variable","Year","Stage","Region","Region_From","Region_To","Tech_Out","Value"]
df = df[[c for c in keep if c in df.columns]].copy()
df["Year"]  = pd.to_numeric(df["Year"], errors="coerce")
df["Value"] = pd.to_numeric(df["Value"], errors="coerce")

In [ ]:
# -------- helpers --------
STAGE_ORDER = ["mine","proc","cath","cell","pack"]

def normalize_stage_from_variable(s: pd.Series, prefix: str) -> pd.Series:
    """
    If Stage is missing/blank, parse it from Variable by regex like:
    demand_pack[...] -> pack ; prod_cell[...] -> cell ; etc.
    """
    pat = rf"^{prefix}_([a-zA-Z0-9]+)"
    extracted = s.str.extract(pat, expand=False)
    return extracted.str.lower()

def ensure_stage(frame: pd.DataFrame, prefix: str) -> pd.DataFrame:
    """
    Ensure a Stage column exists for a given slice (demand/prod/import/export).
    If Stage is NA/blank, fill from Variable string using the prefix supplied.
    """
    g = frame.copy()
    if "Stage" not in g.columns:
        g["Stage"] = pd.NA
    # fill only when missing/blank
    mask = g["Stage"].isna() | (g["Stage"].astype(str).str.strip() == "")
    if mask.any():
        g.loc[mask, "Stage"] = normalize_stage_from_variable(g.loc[mask, "Variable"].astype(str), prefix=prefix)
    # normalize to lowercase and strip
    g["Stage"] = g["Stage"].astype(str).str.strip().str.lower()
    return g

def _sum_by_stage_over_techout(frame: pd.DataFrame,
                               prefix: str,
                               region_col: str | None = None,
                               region_val: str | None = None) -> pd.Series:
    """
    Ensure Stage, optionally filter by region, then:
    if Tech_Out exists -> sum by (Stage, Tech_Out) then across Tech_Out,
    else sum by Stage. Returns Series indexed by Stage.
    """
    g = ensure_stage(frame, prefix=prefix)
    if region_col is not None and region_val is not None and region_col in g.columns:
        g = g[g[region_col].astype(str).str.casefold() == str(region_val).casefold()]
    if g.empty:
        return pd.Series(dtype=float)

    if "Tech_Out" in g.columns:
        step1 = g.groupby(["Stage","Tech_Out"], dropna=False)["Value"].sum(min_count=1).reset_index()
        stage_tot = step1.groupby("Stage", dropna=False)["Value"].sum(min_count=1)
    else:
        stage_tot = g.groupby("Stage", dropna=False)["Value"].sum(min_count=1)

    return stage_tot

def _reindex_stage(s: pd.Series) -> pd.Series:
    return s.reindex(STAGE_ORDER, fill_value=0.0)

# -------- pick the year you want --------
YEAR = 2050  # change this to 2038 or any year you need
d = df[df["Year"] == YEAR].copy()

# Demand
dem_raw = d[d["Variable"].str.startswith("demand_", na=False)]
us_demand    = _reindex_stage(_sum_by_stage_over_techout(dem_raw, prefix="demand", region_col="Region",      region_val="US"))
china_demand = _reindex_stage(_sum_by_stage_over_techout(dem_raw, prefix="demand", region_col="Region",      region_val="China"))
total_demand = us_demand.add(china_demand, fill_value=0.0)
global_demand = total_demand.copy()  # replace with explicit global rows if you have them

# Production
prod_raw = d[d["Variable"].str.startswith("prod_", na=False)]
us_prod    = _reindex_stage(_sum_by_stage_over_techout(prod_raw, prefix="prod", region_col="Region",      region_val="US"))
china_prod = _reindex_stage(_sum_by_stage_over_techout(prod_raw, prefix="prod", region_col="Region",      region_val="China"))

# Imports (destination = Region_To; sum across origins)
imp_raw = d[d["Variable"].str.startswith("import_", na=False)]
us_import    = _reindex_stage(_sum_by_stage_over_techout(imp_raw, prefix="import", region_col="Region_To",  region_val="US"))
china_import = _reindex_stage(_sum_by_stage_over_techout(imp_raw, prefix="import", region_col="Region_To",  region_val="China"))

# Exports (origin = Region_From; sum across destinations) – kept in table so balances work
exp_raw = d[d["Variable"].str.startswith("export_", na=False)]
us_export    = _reindex_stage(_sum_by_stage_over_techout(exp_raw, prefix="export", region_col="Region_From", region_val="US"))
china_export = _reindex_stage(_sum_by_stage_over_techout(exp_raw, prefix="export", region_col="Region_From", region_val="China"))

# -------- assemble table in requested order --------
summary = pd.DataFrame({"Stage": STAGE_ORDER})
summary["Global Demand"] = global_demand.values
summary["US Demand"]     = us_demand.values
summary["China Demand"]  = china_demand.values
summary["Total Demand"]  = summary["US Demand"] + summary["China Demand"]

summary["US Prod"]       = us_prod.values
summary["China Prod"]    = china_prod.values
summary["US Import"]     = us_import.values
summary["China Import"]  = china_import.values
summary["US Export"]     = us_export.values
summary["China Export"]  = china_export.values

# balance constraints: Prod + Import − Export
summary["Balance Constr_US"]    = summary["US Prod"]    + summary["US Import"]    - summary["US Export"]
summary["Balance Constr_China"] = summary["China Prod"] + summary["China Import"] - summary["China Export"]

summary.insert(0, "Year", YEAR)

# -------- show & save to Excel --------
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", None)
print(summary)

out_xlsx = f"balance_summary_{YEAR}.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    summary.to_excel(writer, index=False, sheet_name=f"Y{YEAR}")

print(f"\n✅ Saved: {out_xlsx}")